# ESI HV module activation probe

This notebook isolates the three independent HV conditions exposed by `COM-ESI-CTRL.dll`: target voltage, controller-wide `SetEnable`, and module-level `SetModuleActivationState`. It tests whether the module toggle depends on the controller-wide gate.

## Safety

- Run on the Windows PC connected to the ESI controller.
- Close ESIBD Explorer and ESI-Control first; the DLL is single-instance.
- Keep the hardware interlock available and use an external meter.
- The default configuration never writes a nonzero target. It writes `0 V` to both HV modules, tests the module toggle with the global gate OFF, resets to standby, then repeats with the global gate ON.
- The probe does not restore previous output states. Its finalizer writes `0 V`, requests module standby, disables the global gate, and closes the COM port.
- A nonzero test requires changing `ARM_NONZERO_TEST` to `True`. Start at `10 V`.
- The vendor DLL calls are not cancellable. Restart the Jupyter kernel if a call blocks.


In [ ]:
# Adjust these values only while ESIBD Explorer is closed.
COM_PORT = 16
BAUD_RATE = 230400
MODULE_ADDRESS = 1  # HV1=1, HV2=2
ARM_NONZERO_TEST = False
TEST_VOLTAGE = 10.0
SETTLE_SECONDS = 2.0

from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / 'esi',
    Path(r'C:/Users/lab_admin/ESIBD Explorer/plugins/esi'),
]
PLUGIN_DIR = next(
    (path for path in candidates if (path / 'esi_plugin.py').is_file()),
    None,
)
if PLUGIN_DIR is None:
    raise FileNotFoundError('Could not locate the installed esi plugin folder.')

DRIVER_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'esi' / 'esi_base.py'
DLL_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'esi' / 'vendor' / 'x64' / 'COM-ESI-CTRL.dll'
ERROR_CODES_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'error_codes.json'
print(f'Plugin: {PLUGIN_DIR}')
print(f'COM{COM_PORT} @ {BAUD_RATE} baud; module={MODULE_ADDRESS}')
print(f'Nonzero test armed: {ARM_NONZERO_TEST}')


In [ ]:
import ctypes
import importlib.util
import json
import time
from datetime import datetime

spec = importlib.util.spec_from_file_location('_esi_activation_probe_base', DRIVER_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not load {DRIVER_FILE}')
driver_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(driver_module)
ESIBase = driver_module.ESIBase

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

def record_call(device, action, method, *args, required=False):
    started = time.monotonic()
    try:
        result = method(*args)
        if isinstance(result, tuple):
            status, values = int(result[0]), list(result[1:])
        else:
            status, values = int(result), []
        record = {
            'action': action,
            'status': status,
            'status_text': device.format_status(status),
            'duration_s': round(time.monotonic() - started, 6),
            'values': json_safe(values),
        }
    except Exception as exc:
        record = {
            'action': action,
            'error': f'{type(exc).__name__}: {exc}',
            'duration_s': round(time.monotonic() - started, 6),
        }
        if required:
            raise
        return record
    if required and status != device.NO_ERR:
        raise RuntimeError(f'{action} failed: {device.format_status(status)}')
    return record

def read_module_led(device, address):
    method = getattr(device, 'get_module_led_data', None)
    if callable(method):
        return method(address)
    red = ctypes.c_bool()
    green = ctypes.c_bool()
    blue = ctypes.c_bool()
    function = device.esi_dll.COM_ESI_CTRL_GetModuleLEDData
    function.argtypes = [
        ctypes.c_uint,
        ctypes.POINTER(ctypes.c_bool),
        ctypes.POINTER(ctypes.c_bool),
        ctypes.POINTER(ctypes.c_bool),
    ]
    function.restype = ctypes.c_int
    status = function(
        ctypes.c_uint(address),
        ctypes.byref(red),
        ctypes.byref(green),
        ctypes.byref(blue),
    )
    return status, red.value, green.value, blue.value

def read_hv_state(device, address):
    state = {
        'global_enable': record_call(device, 'get_enable', device.get_enable),
        'controller_activation': record_call(
            device, 'get_activation_state', device.get_activation_state
        ),
        'main_state': record_call(device, 'get_main_state', device.get_main_state),
        'device_state': record_call(device, 'get_device_state', device.get_device_state),
        'interlock_state': record_call(device, 'get_interlock_state', device.get_interlock_state),
        'interlock_enable': record_call(device, 'get_interlock_enable', device.get_interlock_enable),
        'inputs': record_call(device, 'get_inputs', device.get_inputs),
        'power_monitors': record_call(
            device, 'get_power_monitors', device.get_power_monitors
        ),
        'module_state': record_call(device, 'get_module_state', device.get_module_state, address),
        'module_activation_direct': record_call(
            device, 'get_module_activation_state',
            device.get_module_activation_state, address
        ),
        'module_led': record_call(
            device, 'get_module_led_data', read_module_led, device, address
        ),
        'pwm': record_call(device, 'get_hv_supply_params_pwm', device.get_hv_supply_params_pwm, address),
        'target': record_call(
            device, 'get_hv_supply_target_output_voltage',
            device.get_hv_supply_target_output_voltage, address
        ),
        'measured_voltage': record_call(
            device, 'get_hv_supply_output_voltage',
            device.get_hv_supply_output_voltage, address
        ),
        'measured_current': record_call(
            device, 'get_hv_supply_output_current',
            device.get_hv_supply_output_current, address
        ),
    }
    def values_if_ok(name):
        record = state[name]
        return (
            record.get('values', [])
            if record.get('status') == device.NO_ERR else []
        )

    pwm_values = values_if_ok('pwm')
    state['pwm_activation'] = bool(pwm_values[6]) if len(pwm_values) >= 7 else None
    module_state_values = values_if_ok('module_state')
    state['module_state_active'] = (
        bool(int(module_state_values[0]) & device.MS_ACTIVE)
        if module_state_values else None
    )
    target_values = values_if_ok('target')
    state['target_v'] = float(target_values[0]) if target_values else None
    enable_values = values_if_ok('global_enable')
    state['global_enabled'] = bool(enable_values[0]) if enable_values else None
    led_values = values_if_ok('module_led')
    state['module_led_rgb'] = led_values[:3] if len(led_values) >= 3 else None
    print(
        f"global={state['global_enabled']}, "
        f"module_pwm_active={state['pwm_activation']}, "
        f"module_state_active={state['module_state_active']}, "
        f"target={state['target_v']} V, "
        f"LED={state['module_led_rgb']}, "
        f"interlock={state['interlock_state'].get('values')}"
    )
    return state

print('Driver loaded. No COM port has been opened yet.')


## Run the activation probe

At `0 V`, observe whether the module LED changes between the requested STANDBY and ACTIVE states. The direct activation getter may return `-10` on firmware `0x0100`; the decisive independent value is `pwm_activation` from `GetHVsupplyParamsPWM`.


In [ ]:
def run_activation_probe():
    if MODULE_ADDRESS not in (1, 2):
        raise ValueError('MODULE_ADDRESS must be 1 or 2.')
    if not 0.0 <= float(TEST_VOLTAGE) <= 10.0:
        raise ValueError('Keep TEST_VOLTAGE between 0 and 10 V for this probe.')

    device = ESIBase(
        com=COM_PORT,
        idn='esi_hv_activation_probe',
        dll_path=DLL_FILE,
        error_codes_path=ERROR_CODES_FILE,
    )
    opened = False
    report = {
        'timestamp': datetime.now().astimezone().isoformat(),
        'com_port': COM_PORT,
        'module_address': MODULE_ADDRESS,
        'armed_nonzero': bool(ARM_NONZERO_TEST),
        'test_voltage_v': float(TEST_VOLTAGE),
        'steps': {},
        'cleanup': {},
    }
    try:
        report['steps']['open'] = record_call(
            device, 'open_port', device.open_port, COM_PORT, required=True
        )
        opened = True
        report['steps']['baud'] = record_call(
            device, 'set_comspeed', device.set_comspeed, BAUD_RATE, required=True
        )

        for address in (1, 2):
            report['steps'][f'zero_target_{address}'] = record_call(
                device, f'zero_target_{address}',
                device.set_hv_supply_target_output_voltage, address, 0.0,
                required=True,
            )
        report['steps']['zero_heat'] = record_call(
            device, 'zero_heat', device.set_heat_ctrl_heater_temperature, 0.0
        )
        report['steps']['global_off_initial'] = record_call(
            device, 'set_enable(False)', device.set_enable, False, required=True
        )
        time.sleep(SETTLE_SECONDS)
        report['steps']['baseline_global_off'] = read_hv_state(
            device, MODULE_ADDRESS
        )
        standby_off_request = record_call(
            device, 'set_module_activation_state(False) with global OFF',
            device.set_module_activation_state, MODULE_ADDRESS, False
        )
        report['steps']['request_standby_global_off'] = standby_off_request
        time.sleep(SETTLE_SECONDS)
        standby_off_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['standby_global_off_first'] = standby_off_state
        if (
            standby_off_request.get('status') != device.NO_ERR
            or standby_off_state['pwm_activation'] is not False
        ):
            report['steps']['request_standby_global_off_retry'] = record_call(
                device, 'retry set_module_activation_state(False) with global OFF',
                device.set_module_activation_state, MODULE_ADDRESS, False
            )
            time.sleep(SETTLE_SECONDS)
            standby_off_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['standby_global_off_retry'] = standby_off_state
        report['steps']['standby_global_off_final'] = standby_off_state
        if standby_off_state['pwm_activation'] is not False:
            raise RuntimeError('Could not confirm standby with global gate OFF.')
        first_off_request = record_call(
            device, 'set_module_activation_state(True) with global OFF',
            device.set_module_activation_state, MODULE_ADDRESS, True
        )
        report['steps']['request_active_global_off'] = first_off_request
        time.sleep(SETTLE_SECONDS)
        first_off_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['active_global_off_first'] = first_off_state
        off_active_state = first_off_state
        if (
            first_off_request.get('status') != device.NO_ERR
            or first_off_state['pwm_activation'] is not True
        ):
            report['steps']['request_active_global_off_retry'] = record_call(
                device, 'retry set_module_activation_state(True) with global OFF',
                device.set_module_activation_state, MODULE_ADDRESS, True
            )
            time.sleep(SETTLE_SECONDS)
            off_active_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['active_global_off_retry'] = off_active_state
        report['steps']['active_global_off_final'] = off_active_state

        reset_request = record_call(
            device, 'set_module_activation_state(False) before global ON',
            device.set_module_activation_state, MODULE_ADDRESS, False
        )
        report['steps']['request_standby_reset'] = reset_request
        time.sleep(SETTLE_SECONDS)
        reset_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['standby_reset_global_off_first'] = reset_state
        if (
            reset_request.get('status') != device.NO_ERR
            or reset_state['pwm_activation'] is not False
        ):
            report['steps']['request_standby_reset_retry'] = record_call(
                device, 'retry set_module_activation_state(False) before global ON',
                device.set_module_activation_state, MODULE_ADDRESS, False
            )
            time.sleep(SETTLE_SECONDS)
            reset_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['standby_reset_global_off_retry'] = reset_state
        report['steps']['standby_reset_global_off_final'] = reset_state
        if reset_state['pwm_activation'] is not False:
            raise RuntimeError('Could not confirm standby before global-ON test.')
        report['steps']['global_on_for_test'] = record_call(
            device, 'set_enable(True)', device.set_enable, True, required=True
        )
        time.sleep(SETTLE_SECONDS)
        report['steps']['baseline_global_on'] = read_hv_state(
            device, MODULE_ADDRESS
        )
        standby_on_request = record_call(
            device, 'set_module_activation_state(False) with global ON',
            device.set_module_activation_state, MODULE_ADDRESS, False
        )
        report['steps']['request_standby_global_on'] = standby_on_request
        time.sleep(SETTLE_SECONDS)
        standby_on_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['standby_global_on_first'] = standby_on_state
        if (
            standby_on_request.get('status') != device.NO_ERR
            or standby_on_state['pwm_activation'] is not False
        ):
            report['steps']['request_standby_global_on_retry'] = record_call(
                device, 'retry set_module_activation_state(False) with global ON',
                device.set_module_activation_state, MODULE_ADDRESS, False
            )
            time.sleep(SETTLE_SECONDS)
            standby_on_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['standby_global_on_retry'] = standby_on_state
        report['steps']['standby_global_on_final'] = standby_on_state
        if standby_on_state['pwm_activation'] is not False:
            raise RuntimeError('Could not confirm standby with global gate ON.')
        input('Observe the module LED in STANDBY with global ON, then press Enter...')

        first_request = record_call(
            device, 'set_module_activation_state(True) with global ON',
            device.set_module_activation_state, MODULE_ADDRESS, True
        )
        report['steps']['request_active_global_on'] = first_request
        time.sleep(SETTLE_SECONDS)
        first_active_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['active_global_on_first'] = first_active_state
        active_state = first_active_state
        if (
            first_request.get('status') != device.NO_ERR
            or first_active_state['pwm_activation'] is not True
        ):
            report['steps']['request_active_global_on_retry'] = record_call(
                device, 'retry set_module_activation_state(True) with global ON',
                device.set_module_activation_state, MODULE_ADDRESS, True
            )
            time.sleep(SETTLE_SECONDS)
            active_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['active_global_on_retry'] = active_state
        report['steps']['active_zero_v_final'] = active_state
        input('Observe the module LED after ACTIVE requests at 0 V, then press Enter...')

        if ARM_NONZERO_TEST:
            if active_state['pwm_activation'] is not True:
                raise RuntimeError('PWM status did not confirm module activation.')
            if active_state['global_enabled'] is not True:
                raise RuntimeError('Global enable readback is not ON.')
            report['steps']['set_test_voltage'] = record_call(
                device, 'set_test_voltage',
                device.set_hv_supply_target_output_voltage,
                MODULE_ADDRESS, float(TEST_VOLTAGE), required=True,
            )
            time.sleep(SETTLE_SECONDS)
            report['steps']['nonzero'] = read_hv_state(device, MODULE_ADDRESS)
            input('Record both physical output voltages, then press Enter to force OFF...')

        return report
    finally:
        if opened:
            for address in (1, 2):
                report['cleanup'][f'zero_target_{address}'] = record_call(
                    device, f'cleanup_zero_target_{address}',
                    device.set_hv_supply_target_output_voltage, address, 0.0
                )
                standby = record_call(
                    device, f'cleanup_standby_{address}',
                    device.set_module_activation_state, address, False
                )
                report['cleanup'][f'standby_{address}'] = standby
                if standby.get('status') != device.NO_ERR:
                    report['cleanup'][f'standby_retry_{address}'] = record_call(
                        device, f'cleanup_standby_retry_{address}',
                        device.set_module_activation_state, address, False
                    )
            report['cleanup']['global_off'] = record_call(
                device, 'cleanup_global_off', device.set_enable, False
            )
            try:
                time.sleep(SETTLE_SECONDS)
                report['cleanup']['final_state'] = read_hv_state(
                    device, MODULE_ADDRESS
                )
            except Exception as exc:
                report['cleanup']['final_state'] = {
                    'error': f'{type(exc).__name__}: {exc}'
                }
            finally:
                report['cleanup']['close'] = record_call(
                    device, 'close_port', device.close_port
                )
        report_path = PLUGIN_DIR / f"esi_hv_activation_probe_{datetime.now():%Y%m%d_%H%M%S}.json"
        report_path.write_text(
            json.dumps(json_safe(report), indent=2), encoding='utf-8'
        )
        print(f'Report written to {report_path}')

activation_report = run_activation_probe()


## Interpretation

- `active_global_off_final.pwm_activation == true`: the global gate is not a prerequisite for changing the HVC toggle.
- `active_global_off_final.pwm_activation == false` followed by `baseline_global_on.pwm_activation == true`: the OFF-gate request was latched but the PWM readback hides it until `SetEnable(True)`.
- `active_global_off_final.pwm_activation == false`, `baseline_global_on.pwm_activation == false`, then `active_zero_v_final.pwm_activation == true`: `SetEnable(True)` must precede module activation; the plugin sequence must be changed.
- An ACTIVE request returning `-10` with `active_zero_v_final.pwm_activation == true`: firmware omitted the direct acknowledgement, but the HVC toggle was applied.
- `active_zero_v_final.pwm_activation` remains false after both global-ON attempts: the command was not applied. Do not arm the nonzero test; inspect `interlock_state`, `interlock_enable`, `inputs`, `controller_activation`, `module_state_active`, and `module_led_rgb`.
- `pwm_activation == true`, `global_enabled == true`, and target readback is correct, but physical voltage remains zero: inspect interlocks and input levels, then verify both physical outputs externally.
- Yellow in both observed states is consistent with the module remaining inhibited.
